# Getting Started

In this demo, we show you how to:
* Wrap an MLP in a PyTorch model with our `MLP_SR` class
* Perform symbolic regresson on the MLP with the `distill` method 
* Switch the MLP to an equation in the forward pass of the model with the `switch_to_equation` method 
* Switch back to the MLP with the `switch_to_MLP` method

## Wrapping a PyTorch model
Create a simple PyTorch model.

In [1]:
import torch
import numpy as np
import torch.nn as nn

class MLP(nn.Module):
    """
    Simple MLP.
    """
    def __init__(self, input_dim, output_dim, hidden_dim):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

class SimpleModel(nn.Module):
    """
    Simple model class.
    """
    def __init__(self, input_dim, output_dim, hidden_dim = 64):
        super(SimpleModel, self).__init__()

        self.mlp = MLP(input_dim, output_dim, hidden_dim)

    def forward(self, x):
        x = self.mlp(x)
        return x

Train the model on some data.

$$
y = x_0^2 +3 \sin(x_4)-4
$$

In [2]:
# Make the dataset 
x = np.array([np.random.uniform(0, 1, 10_000) for _ in range(5)]).T  
y = x[:, 0]**2 + 3*np.sin(x[:, 4]) - 4
noise = np.array([np.random.normal(0, 0.05*np.std(y)) for _ in range(len(y))])
y = y + noise 

In [3]:
# Set up training

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

def train_model(model, dataloader, opt, criterion, epochs = 100):
    """
    Train a model for the specified number of epochs.
    
    Args:
        model: PyTorch model to train
        dataloader: DataLoader for training data
        opt: Optimizer
        criterion: Loss function
        epochs: Number of training epochs
        
    Returns:
        tuple: (trained_model, loss_tracker)
    """
    loss_tracker = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_x, batch_y in dataloader:
            # Forward pass
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            
            # Backward pass
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            epoch_loss += loss.item()
        
        loss_tracker.append(epoch_loss)
        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / len(dataloader)
            print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_loss:.6f}')
    return model, loss_tracker

# Instantiate the model
model = SimpleModel(input_dim=x.shape[1], output_dim=1)

# Set up training
criterion = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=0.001)
X_train, _, y_train, _ = train_test_split(x, y.reshape(-1,1), test_size=0.2, random_state=290402)

# Set up dataset
dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [4]:
# Train the model and save the weights

model, losses = train_model(model, dataloader, opt, criterion, 20)
torch.save(model.state_dict(), 'model_weights.pth')

Epoch [5/20], Avg Loss: 0.073434
Epoch [10/20], Avg Loss: 0.053065
Epoch [15/20], Avg Loss: 0.042182
Epoch [20/20], Avg Loss: 0.035459


Wrap the mlp layer in the trained model with MLP_SR.

In [14]:
from symtorch import MLP_SR
model.mlp = MLP_SR(model.mlp, mlp_name = 'Sequential')

## Interpret the MLP

In this example, we pass extra parameters into the `.distill` method (complexity of operators/constants and parsimony, which is a penalisation of complexity).\
To see all the possible parameters, please see the `PySRRegressor` class from [PySR](https://ai.damtp.cam.ac.uk/pysr/api/).

In this example, we turn verbosity off because we are in a Jupyter notebook. For best performance, run in IPython, as you can terminate the SR any time.

In [ ]:
# Configure the SR

sr_params = {'complexity_of_operators':  {"sin":3, "exp":3},
             'complexity_of_constants': 2, 
             'parsimony': 0.1,
             'verbosity': 0,
             'niterations': 500}

model.mlp.distill(torch.FloatTensor(X_train),
                       sr_params = sr_params, 
                       parent_model=model) #Pass in the parent model (really only required if the MLP is 
                                            #not at the start of the model but it is good practice)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


💡Best equation for output 0 found to be (sin(x0 * x0) + -3.9160948) + (sin(x4) * 2.7672863).
❤️ SR on Sequential complete.


{0: PySRRegressor.equations_ = [
 	    pick         score                                           equation  \
 	0         0.000000e+00                                                 x4   
 	1         2.721272e+00                                         -2.3407037   
 	2         4.214388e-01                                    x4 + -2.8381717   
 	3         4.685426e-01                             (x4 + x4) + -3.3356333   
 	4         1.191554e-01                       (x4 + -1.4930925) * 2.350959   
 	5         1.376663e+00                      (x0 + x4) + (x4 + -3.8348954)   
 	6         7.879721e-01               ((x4 * 2.3659413) + -4.0169363) + x0   
 	7         1.743718e-01        ((x4 * 2.3639326) + -3.8495965) + (x0 * x0)   
 	8         2.638426e-01           ((sin(x4) * 2.770242) + -4.1069636) + x0   
 	9         3.600058e-01   (x0 * x0) + ((sin(x4) * 2.7674358) + -3.9393387)   
 	10  >>>>  1.106949e-01  (sin(x0 * x0) + -3.9160948) + (sin(x4) * 2.767...   
 	11        7.00703

See the full Pareto front of equations. The best equation is chosen as a balance of accuracy and complexity.\
Outputs from *PySR* are saved in `SR_output/MLP_name`.

In [7]:
print(model.mlp.pysr_regressor)

{0: PySRRegressor.equations_ = [
	    pick         score                                           equation  \
	0         0.000000e+00                                                 x4   
	1         2.721272e+00                                         -2.3407037   
	2         4.214388e-01                                    x4 + -2.8381717   
	3         4.685426e-01                             (x4 + x4) + -3.3356333   
	4         1.191554e-01                       (x4 + -1.4930925) * 2.350959   
	5         1.376663e+00                      (x0 + x4) + (x4 + -3.8348954)   
	6         7.879721e-01               ((x4 * 2.3659413) + -4.0169363) + x0   
	7         1.743718e-01        ((x4 * 2.3639326) + -3.8495965) + (x0 * x0)   
	8         2.638426e-01           ((sin(x4) * 2.770242) + -4.1069636) + x0   
	9         3.600058e-01   (x0 * x0) + ((sin(x4) * 2.7674358) + -3.9393387)   
	10  >>>>  1.106949e-01  (sin(x0 * x0) + -3.9160948) + (sin(x4) * 2.767...   
	11        7.007036e-02  (sin(x

## Switch to Using the Equation Instead in the Forwards Pass

You can choose the equation you want to switch to by choosing the desired complexity of equation. \
If left blank, then we choose the best equation chosen by *PySR*.

In [8]:
model.mlp.switch_to_equation(complexity=[14]) 

✅ Successfully switched Sequential to symbolic equations for all 1 dimensions:
   Dimension 0: (x0 * x0) + ((sin(x4) * 2.7674358) + -3.9393387)
   Variables: ['x0', 'x4']
🎯 All 1 output dimensions now using symbolic equations.


Now when running the forwards pass through the model, it uses the symbolic equation instead of the MLP. 

In [9]:
interpretable_outputs = model(torch.tensor(X_train, dtype=torch.float32))
interpretable_outputs

tensor([[-2.9064],
        [-2.0674],
        [-1.7499],
        ...,
        [-1.7663],
        [-2.5468],
        [-3.3291]])

## Switch to Using the MLP in the Forwards Pass

In [10]:
mlp_outputs = model.mlp.switch_to_mlp()

✅ Switched Sequential back to MLP


In [11]:
model.eval()
with torch.no_grad():
    model_outputs = model.mlp(torch.tensor(X_train, dtype=torch.float32))
model_outputs

tensor([[-2.9956],
        [-2.0082],
        [-1.7034],
        ...,
        [-1.7170],
        [-2.5529],
        [-3.3298]])

In [12]:
model_outputs.shape

torch.Size([8000, 1])

In [13]:
# Clean up 
import os
import shutil
if os.path.exists('SR_output'):
    shutil.rmtree('SR_output')
os.remove('model_weights.pth')